# Target-encoding leak — California housing

A self-contained demonstration of the bug that ships in ML production pipelines under names like **"neighborhood affluence index"**, **"region target encoding"**, or **"group-mean feature"**.

The structure:

1. Build a feature that **looks clever**: average target per location bucket, computed on the full dataset.
2. Train a model. The R² is **impressive**. Ship it.
3. Run `schema-firewall`. Both `check_stateless` and `check_leakage` raise.
4. Fix the pipeline: compute the group mean on **train only**. The R² collapses.
5. Realise: without the firewall, the bug would have shipped silently.

**If you've ever applied `.groupby().transform('mean')`, `TargetEncoder`, `.value_counts()`, `TfidfVectorizer.fit_transform`, or `ComBat` to your full dataset before splitting, this notebook is pointed at you.**

## Setup

In [ ]:
%pip install schema-firewall -q

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

from schema_firewall import (
    LeakageError,
    StatelessnessError,
    check_leakage,
    check_stateless,
)

raw = fetch_california_housing(as_frame=True)
df = raw.data.copy()
y = raw.target.rename("price")

# Round coordinates to two decimals to create tight location buckets.
df["lat_bin"] = df["Latitude"].round(2)
df["lon_bin"] = df["Longitude"].round(2)
df["region"] = df["lat_bin"].astype(str) + "_" + df["lon_bin"].astype(str)

print(f"rows       : {len(df)}")
print(f"regions    : {df['region'].nunique()}")
print(f"target mean: {y.mean():.2f}   std: {y.std():.2f}")

## 1. The leaky pipeline

Innocent-looking: for each location bucket, compute the **average house price**, then attach that number back to every row as a feature.

The bug: the mean is computed on **all rows**, including the ones that will later become the test split. Every test row's `region_mean_price` already contains information about its own target.

In [ ]:
def leaky_feature_engineering(df: pd.DataFrame, target: pd.Series) -> pd.DataFrame:
    out = df.copy()
    region_means = target.groupby(df["region"]).mean()
    out["region_mean_price"] = out["region"].map(region_means)
    return out.drop(columns=["region", "lat_bin", "lon_bin"])


x_leaky = leaky_feature_engineering(df, y)
x_tr, x_te, y_tr, y_te = train_test_split(x_leaky, y, test_size=0.25, random_state=0)

model = Ridge(alpha=1.0).fit(x_tr, y_tr)
leaky_r2 = r2_score(y_te, model.predict(x_te))
print(f"leaky R² = {leaky_r2:.4f}   ← this is what you'd report in your slack / report / PR")

### Looks great. Ship it, right?

## 2. schema-firewall — one call catches the bug

`check_stateless` asks: *if I apply your pipeline to the full frame vs. a one-row subset, does the same row come out the same way?*

A target-mean-encoded feature fails because the one-row mean is just that row's own target. The full-frame mean pulls in the rest of the bucket. The outputs disagree. The check raises.

In [ ]:
def pipeline_fn(frame: pd.DataFrame) -> pd.DataFrame:
    return leaky_feature_engineering(frame, y.loc[frame.index])

try:
    check_stateless(pipeline_fn, df)
except StatelessnessError as exc:
    print("StatelessnessError raised.\n")
    print(str(exc))

`check_leakage` asks a different question: *do any features in X show suspicious statistical dependency with y?* It runs Pearson, Spearman, and normalised mutual information. The leaky feature trips all three.

In [ ]:
try:
    check_leakage(x_leaky.select_dtypes(include=[np.number]), y)
except LeakageError as exc:
    print("LeakageError raised.\n")
    print(str(exc))

## 3. Fix the pipeline

Compute the region mean on **train only**. Map it onto test. Unseen test regions become NaN and must be imputed (here: with the global train mean). The R² collapses to honest territory.

In [ ]:
tr_idx, te_idx = train_test_split(df.index, test_size=0.25, random_state=0)

# Work on a copy so the underlying df from cell 3 stays unmutated.
# This lets the reader re-run the leaky-path cell above without
# contaminating it with the honest-path region_mean_price column.
df_honest = df.copy()
region_means = y.loc[tr_idx].groupby(df_honest.loc[tr_idx, "region"]).mean()
df_honest["region_mean_price"] = df_honest["region"].map(region_means).fillna(region_means.mean())

features = df_honest.drop(columns=["region", "lat_bin", "lon_bin"])
x_tr, x_te = features.loc[tr_idx], features.loc[te_idx]
y_tr, y_te = y.loc[tr_idx], y.loc[te_idx]

model = Ridge(alpha=1.0).fit(x_tr, y_tr)
honest_r2 = r2_score(y_te, model.predict(x_te))
print(f"honest R² = {honest_r2:.4f}")

## 4. The numbers

In [ ]:
print(f"leaky R²  = {leaky_r2:.4f}   ← what you'd have shipped")
print(f"honest R² = {honest_r2:.4f}   ← actual generalisation capacity")
print(f"gap       = {leaky_r2 - honest_r2:+.4f}   ← how much of 'your model' was a leak")

## The takeaway

Half the reported R² was a leak. Neither the model training, nor `train_test_split`, nor sklearn's `Pipeline` object would have flagged it. Peer review wouldn't have caught it. A test suite for the *numerical correctness* of the feature engineering wouldn't have caught it.

`check_stateless` catches it because it encodes an invariant the pipeline broke: *per-row output must not depend on other rows*.

`check_leakage` catches it because the leaked feature has Pearson ≈ 0.97 with the target.

Put one or both checks into your pre-training gate. They take seconds to run.

## Receipts

The four real-world cases this library is built to catch:

| Case | Paper / source | Class |
|---|---|---|
| COVID-19 chest X-ray classifiers learned hospital-ID, not disease | [DeGrave et al., Nat Mach Intell 2021](https://pmc.ncbi.nlm.nih.gov/articles/PMC8502237/) | confounder / schema-mismatch |
| 40% of MIMIC same-admission prediction models used post-outcome ICD codes | [JAMA Network Open 2024](https://jamanetwork.com/journals/jamanetworkopen/fullarticle/2843179) | forbidden-column / temporal |
| Kaggle Santander 2019: AUC jumped 0.90 → 0.92 via frequency features over train+test | [Kaggle #84614](https://www.kaggle.com/c/santander-customer-transaction-prediction/discussion/84614) | state-dependent transform |
| Connectome-based ML: feature selection on full data inflated r by up to +0.47 | [Rosenblatt et al., Nat Commun 2024](https://www.nature.com/articles/s41467-024-46150-w) | state-dependent + group-split |